In [11]:
!pip install --upgrade gdown
import gdown

folder_url = 'https://drive.google.com/drive/folders/1s5W0Nfz9NEN8gP14tge8GuouUDXs2Ssq'
gdown.download_folder(folder_url, quiet=False, use_cookies=False)

Retrieving folder contents


Processing file 1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF full-shape.zip
Processing file 1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p partial.zip
Processing file 1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM rotate.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF
From (redirected): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF&confirm=t&uuid=b16fdd50-9045-4834-a63a-f72ea77f398a
To: /content/AffordanceNet/full-shape.zip
100%|██████████| 558M/558M [00:04<00:00, 122MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p
From (redirected): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p&confirm=t&uuid=7715073e-a528-40bf-ae85-a5fedfe87358
To: /content/AffordanceNet/partial.zip
100%|██████████| 2.19G/2.19G [00:19<00:00, 114MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM
To: /content/AffordanceNet/rotate.zip
100%|██████████| 1.15M/1.15M [00:00<00:00, 78.4MB/s]
Download completed


['/content/AffordanceNet/full-shape.zip',
 '/content/AffordanceNet/partial.zip',
 '/content/AffordanceNet/rotate.zip']

In [2]:
# Unzip all files in the AffordanceNet folder
!unzip -o /content/AffordanceNet/full-shape.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/partial.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/rotate.zip -d /content/AffordanceNet/

# List the directory to see the extracted content
!ls -R /content/AffordanceNet/

Archive:  /content/AffordanceNet/full-shape.zip
  inflating: /content/AffordanceNet/full_shape_train_data.pkl  
  inflating: /content/AffordanceNet/full_shape_val_data.pkl  
Archive:  /content/AffordanceNet/partial.zip
  inflating: /content/AffordanceNet/partial_train_data.pkl  
  inflating: /content/AffordanceNet/partial_val_data.pkl  
Archive:  /content/AffordanceNet/rotate.zip
  inflating: /content/AffordanceNet/rotate_val_data.pkl  
/content/AffordanceNet/:
full_shape_train_data.pkl  partial_train_data.pkl  rotate_val_data.pkl
full_shape_val_data.pkl    partial_val_data.pkl    rotate.zip
full-shape.zip		   partial.zip


In [3]:
# Remove all zip files in the AffordanceNet directory
!rm /content/AffordanceNet/*.zip

# List the directory to confirm they are gone
!ls /content/AffordanceNet/

full_shape_train_data.pkl  partial_train_data.pkl  rotate_val_data.pkl
full_shape_val_data.pkl    partial_val_data.pkl


In [4]:
!pip install torch torchvision tqdm numpy

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pickle
from tqdm import tqdm

# ---------------------------
# Affordance & data constants
# ---------------------------
AFFORDANCE_NAMES = ['grasp', 'contain', 'lift', 'openable', 'layable', 'sittable', 'support',
                    'wrap_grasp', 'pourable', 'move', 'displaY', 'pushable', 'pull', 'listen',
                    'wear', 'press', 'cut', 'stab']
SELECTED_CLASSES = ['move', 'support', 'sittable', 'contain', 'displaY']
SELECTED_INDICES = [AFFORDANCE_NAMES.index(c) for c in SELECTED_CLASSES]
NUM_AFFORDANCES = len(SELECTED_CLASSES)
NUM_PARTS = 7  # body, handle, blade, flat, concave, rim, display

In [6]:
def knn(x, k=20):
    B, _, N = x.shape
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    idx = pairwise_distance.topk(k=k, dim=-1)[1]
    return idx

def get_graph_feature(x, k=20, idx=None):
    B, C, N = x.shape
    if idx is None:
        idx = knn(x, k=k)
    device = x.device
    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx = idx + idx_base
    idx = idx.view(-1)
    _, num_dims, _ = x.size()
    x = x.transpose(2, 1).contiguous()
    feature = x.view(B * N, -1)[idx, :]
    feature = feature.view(B, N, k, C)
    x = x.view(B, N, 1, C).repeat(1, 1, k, 1)
    feature = torch.cat((feature - x, x), dim=3)
    feature = feature.permute(0, 3, 1, 2).contiguous()
    return feature

In [7]:
class MultiTaskDGCNN(nn.Module):
    def __init__(self, num_affordances=5, num_parts=7, k=20):
        super().__init__()
        self.k = k
        self.conv1 = nn.Sequential(nn.Conv2d(6, 64, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU())
        self.conv2 = nn.Sequential(nn.Conv2d(64*2, 64, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU())
        self.conv3 = nn.Sequential(nn.Conv2d(64*2, 128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU())
        self.fc_shared = nn.Sequential(
            nn.Conv1d(64+64+128+256, 256, 1, bias=False), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 128, 1, bias=False), nn.BatchNorm1d(128), nn.ReLU()
        )
        self.affordance_head = nn.Conv1d(128, num_affordances, 1)
        self.part_head = nn.Conv1d(128, num_parts, 1)

    def forward(self, x):
        B, C, N = x.shape
        edge_feat = get_graph_feature(x, k=self.k)
        x1 = self.conv1(edge_feat).max(dim=-1)[0]
        edge_feat = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(edge_feat).max(dim=-1)[0]
        edge_feat = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(edge_feat).max(dim=-1)[0]
        x_cat = torch.cat([x1, x2, x3], dim=1)
        x_global = x_cat.max(dim=-1, keepdim=True)[0].repeat(1,1,N)
        x_concat = torch.cat([x1, x2, x3, x_global], dim=1)
        shared_feat = self.fc_shared(x_concat)
        aff_logits = self.affordance_head(shared_feat).permute(0,2,1)
        part_logits = self.part_head(shared_feat).permute(0,2,1)
        return aff_logits, part_logits

In [8]:
class MultiTaskLoss(nn.Module):
    def __init__(self, aff_weight=1.0, part_weight=0.5, dice_weight=1.0, smooth=1.0):
        super().__init__()
        self.aff_weight = aff_weight
        self.part_weight = part_weight
        self.dice_weight = dice_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, aff_logits, part_logits, aff_targets, part_targets):
        bce_loss = self.bce(aff_logits, aff_targets)
        aff_probs = torch.sigmoid(aff_logits)
        dice_loss = 0.0
        for c in range(aff_targets.size(-1)):
            inter = (aff_probs[..., c] * aff_targets[..., c]).sum()
            union = aff_probs[..., c].sum() + aff_targets[..., c].sum()
            dice = (2. * inter + self.smooth) / (union + self.smooth)
            dice_loss += (1 - dice)
        dice_loss /= aff_targets.size(-1)
        aff_loss = bce_loss + self.dice_weight * dice_loss
        part_loss = F.cross_entropy(part_logits.permute(0,2,1), part_targets)
        total = self.aff_weight * aff_loss + self.part_weight * part_loss
        return total, aff_loss.item(), part_loss.item()

In [9]:
class MultiTaskAffordanceDataset(Dataset):
    def __init__(self, data_list, selected_indices, part_labels_list,
                 num_points=2048, occlusion=True, crop_angle=120, dropout=0.3):
        self.data = data_list
        self.selected_indices = selected_indices
        self.part_labels = part_labels_list   # list of int arrays, same length as data_list
        self.num_points = num_points
        self.occlusion = occlusion
        self.crop_angle = crop_angle
        self.dropout = dropout

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        pts = sample['full_shape']['coordinate'].astype(np.float32)
        label_dict = sample['full_shape']['label']

        # Affordance labels
        aff_masks = []
        for aff_idx in self.selected_indices:
            mask = label_dict[AFFORDANCE_NAMES[aff_idx]].flatten()
            mask = (mask > 0.5).astype(np.float32)
            aff_masks.append(mask)
        aff_labels = np.stack(aff_masks, axis=1)

        part_labels = self.part_labels[idx].copy()

        if self.occlusion:
            pts, aff_labels, part_labels = self._occlude(pts, aff_labels, part_labels)
            if len(pts) == 0:
                pts = sample['full_shape']['coordinate'].astype(np.float32)
                aff_labels = np.stack(aff_masks, axis=1)
                part_labels = self.part_labels[idx].copy()

        # Normalise
        centroid = np.mean(pts, axis=0)
        pts -= centroid
        max_dist = np.max(np.linalg.norm(pts, axis=1))
        pts /= (max_dist + 1e-8)

        # Subsample
        if len(pts) > self.num_points:
            pos_mask = (aff_labels.sum(axis=1) > 0.5) | (part_labels > 0)
            pos_idx = np.where(pos_mask)[0]
            neg_idx = np.where(~pos_mask)[0]
            n_pos = len(pos_idx)
            n_neg_needed = max(0, self.num_points - n_pos)
            if n_neg_needed > 0 and len(neg_idx) > 0:
                neg_sel = np.random.choice(neg_idx, n_neg_needed, replace=False)
                chosen = np.concatenate([pos_idx, neg_sel])
            elif n_pos > 0:
                chosen = pos_idx
            else:
                chosen = np.random.choice(len(pts), self.num_points, replace=False)
            if len(chosen) < self.num_points:
                extra = np.random.choice(len(pts), self.num_points - len(chosen), replace=True)
                chosen = np.concatenate([chosen, extra])
        else:
            chosen = np.random.choice(len(pts), self.num_points, replace=True)

        pts = pts[chosen]
        aff_labels = aff_labels[chosen]
        part_labels = part_labels[chosen]

        return (torch.tensor(pts, dtype=torch.float32),
                torch.tensor(aff_labels, dtype=torch.float32),
                torch.tensor(part_labels, dtype=torch.long))

    def _occlude(self, points, aff_labels, part_labels):
        pos_mask = (aff_labels.sum(axis=1) > 0.5) | (part_labels > 0)
        pos_idx = np.where(pos_mask)[0]
        neg_idx = np.where(~pos_mask)[0]
        if len(neg_idx) == 0:
            return points, aff_labels, part_labels
        neg_pts = points[neg_idx]
        centroid = np.mean(neg_pts, axis=0)
        centered = neg_pts - centroid
        theta = np.random.uniform(0, 2*np.pi)
        phi = np.random.uniform(0, np.pi)
        view_dir = np.array([np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi)])
        norms = np.linalg.norm(centered, axis=1); norms[norms==0]=1e-8
        dirs = centered / norms[:, None]
        angles = np.arccos(np.clip(np.dot(dirs, view_dir), -1, 1))
        keep = angles < np.radians(self.crop_angle/2)
        if self.dropout > 0:
            keep_idx = np.where(keep)[0]
            n_keep = int(len(keep_idx)*(1-self.dropout))
            if n_keep>0 and len(keep_idx)>0:
                keep_idx = np.random.choice(keep_idx, n_keep, replace=False)
                keep = np.zeros(len(neg_pts), dtype=bool); keep[keep_idx]=True
        neg_pts = neg_pts[keep]
        neg_aff = aff_labels[neg_idx][keep]
        neg_part = part_labels[neg_idx][keep]
        return np.concatenate([points[pos_idx], neg_pts], 0), \
               np.concatenate([aff_labels[pos_idx], neg_aff], 0), \
               np.concatenate([part_labels[pos_idx], neg_part], 0)

In [13]:
# Load real training data
with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)
all_cats = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = ['Scissors', 'Hammer', 'Laptop', 'Camera']
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_cats]
train_data_real = [s for s in train_data if s['semantic class'] not in TEST_CATEGORIES]
test_split = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES]

# Load geometric part labels (pre‑computed)
with open("geom_real_dataset.pkl", "rb") as f:
    geom_data = pickle.load(f)
part_labels_list = [lbl for _, lbl in geom_data]  # list of int arrays

# Split train/val (90/10)
np.random.seed(42)
idx = np.random.permutation(len(train_data_real))
split = int(0.9 * len(train_data_real))
train_idx = idx[:split]
val_idx = idx[split:]

train_samples = [train_data_real[i] for i in train_idx]
train_parts   = [part_labels_list[i] for i in train_idx]
val_samples   = [train_data_real[i] for i in val_idx]
val_parts     = [part_labels_list[i] for i in val_idx]

# Create datasets (no occlusion for now – you can change to True later)
train_dataset = MultiTaskAffordanceDataset(train_samples, SELECTED_INDICES, train_parts,
                                           num_points=2048, occlusion=False)
val_dataset   = MultiTaskAffordanceDataset(val_samples, SELECTED_INDICES, val_parts,
                                           num_points=2048, occlusion=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

UnpicklingError: pickle data was truncated

In [14]:
import numpy as np
import torch
from tqdm import tqdm

# ---- GPU‑accelerated geometric labelling (same as before) ----
def compute_geometric_labels_gpu(points_batch, k=30):
    B, N, _ = points_batch.shape
    device = points_batch.device
    k = min(k, N-1)
    if k < 1:
        return torch.zeros((B, N), dtype=torch.long, device=device)

    dists = torch.cdist(points_batch, points_batch)
    topk_vals, topk_idx = dists.topk(k=k+1, dim=-1, largest=False)
    neighbor_idx = topk_idx[:, :, 1:]
    neighbors = torch.gather(points_batch.unsqueeze(2).expand(-1, -1, k, -1),
                             dim=1, index=neighbor_idx.unsqueeze(-1).expand(-1, -1, -1, 3))
    centered = neighbors - neighbors.mean(dim=2, keepdim=True)
    cov = torch.matmul(centered.transpose(2, 3), centered) / k
    eigvals, eigvecs = torch.linalg.eigh(cov)
    eigvals = eigvals.flip(-1)
    eigvecs = eigvecs.flip(-1)

    lambda1 = eigvals[:,:,0]; lambda2 = eigvals[:,:,1]; lambda3 = eigvals[:,:,2]
    sum_l = lambda1 + lambda2 + lambda3 + 1e-10
    linearity  = (lambda1 - lambda2) / sum_l
    planarity  = (lambda2 - lambda3) / sum_l
    scattering = lambda3 / sum_l

    normal = eigvecs[:,:,:,2]
    dists_to_plane = torch.abs(torch.sum(centered * normal.unsqueeze(2), dim=-1))
    curvature = dists_to_plane.mean(dim=-1)

    labels = torch.zeros((B, N), dtype=torch.long, device=device)
    mask_disp = (planarity > 0.6) & (curvature < 0.02) & (torch.abs(normal[..., 2]) < 0.3) & (lambda1 > 0.1)
    labels[mask_disp] = 6
    mask_flat = (planarity > 0.6) & (curvature < 0.02) & ~mask_disp
    labels[mask_flat] = 3
    mask_handle = (linearity > 0.6) & (lambda1 > 0.05) & (labels == 0)
    labels[mask_handle] = 1
    mask_concave = (curvature > 0.03) & (planarity < 0.3) & (labels == 0)
    labels[mask_concave] = 4
    mask_blade = (planarity > 0.4) & (curvature > 0.02) & (labels == 0)
    labels[mask_blade] = 2
    return labels

# Load training data and normalise
with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)

TEST_CATEGORIES = ['Scissors', 'Hammer', 'Laptop', 'Camera']
all_cats = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_cats]
train_data_real = [s for s in train_data if s['semantic class'] not in TEST_CATEGORIES]

norm_points = []
for sample in tqdm(train_data_real, desc="Normalising"):
    pts = sample['full_shape']['coordinate'].astype(np.float32)
    centroid = np.mean(pts, axis=0)
    pts -= centroid
    max_dist = np.max(np.linalg.norm(pts, axis=1))
    pts /= (max_dist + 1e-8)
    norm_points.append(pts)

device = torch.device('cuda')
geom_dataset = []
for pts in tqdm(norm_points, desc="GPU labelling"):
    tensor = torch.from_numpy(pts).unsqueeze(0).to(device)
    with torch.no_grad():
        lbl_batch = compute_geometric_labels_gpu(tensor, k=30)
    labels = lbl_batch.squeeze(0).cpu().numpy()
    geom_dataset.append((pts, labels))

with open("geom_real_dataset.pkl", "wb") as f:
    pickle.dump(geom_dataset, f)
print(f"Saved {len(geom_dataset)} labelled point clouds.")

GPU labelling: 100%|██████████| 15738/15738 [00:47<00:00, 331.58it/s]


Saved 15738 labelled point clouds.


In [15]:
# Now load the freshly created geom_real_dataset.pkl
with open("geom_real_dataset.pkl", "rb") as f:
    geom_data = pickle.load(f)
part_labels_list = [lbl for _, lbl in geom_data]

# Re‑define train_data_real (just in case)
train_data_real = [s for s in train_data if s['semantic class'] not in TEST_CATEGORIES]
test_split = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES]

# Split train/val
np.random.seed(42)
idx = np.random.permutation(len(train_data_real))
split = int(0.9 * len(train_data_real))
train_idx = idx[:split]
val_idx = idx[split:]

train_samples = [train_data_real[i] for i in train_idx]
train_parts   = [part_labels_list[i] for i in train_idx]
val_samples   = [train_data_real[i] for i in val_idx]
val_parts     = [part_labels_list[i] for i in val_idx]

# Create datasets
train_dataset = MultiTaskAffordanceDataset(train_samples, SELECTED_INDICES, train_parts,
                                           num_points=2048, occlusion=False)
val_dataset   = MultiTaskAffordanceDataset(val_samples, SELECTED_INDICES, val_parts,
                                           num_points=2048, occlusion=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
print("DataLoaders ready.")

DataLoaders ready.


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiTaskDGCNN(num_affordances=5, num_parts=7, k=20).to(device)
criterion = MultiTaskLoss(aff_weight=1.0, part_weight=0.3, dice_weight=1.0)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

NUM_EPOCHS = 30
best_miou = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for pts, aff, part in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        pts = pts.permute(0,2,1).to(device)
        aff, part = aff.to(device), part.to(device)
        optimizer.zero_grad()
        aff_logits, part_logits = model(pts)
        loss, _, _ = criterion(aff_logits, part_logits, aff, part)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    iou_c = torch.zeros(5).to(device); cnt_c = torch.zeros(5).to(device)
    with torch.no_grad():
        for pts, aff, _ in val_loader:
            pts = pts.permute(0,2,1).to(device); aff = aff.to(device)
            aff_logits, _ = model(pts)
            pred = (torch.sigmoid(aff_logits) > 0.5).float()
            for b in range(pts.size(0)):
                for c in range(5):
                    if aff[b,:,c].sum()>0:
                        inter = (pred[b,:,c] * aff[b,:,c]).sum()
                        union = (pred[b,:,c] + aff[b,:,c]).clamp(0,1).sum()
                        iou = inter / (union + 1e-8)
                        iou_c[c] += iou; cnt_c[c] += 1
    avg_iou = iou_c / (cnt_c + 1e-8)
    miou = avg_iou.mean().item()
    print(f"Epoch {epoch+1}: Loss {avg_loss:.4f}, Val mIoU {miou:.4f}")
    print(f"  Per-class: {[f'{x:.3f}' for x in avg_iou.tolist()]}")
    scheduler.step()
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), "best_multitask.pth")
        print(f"  -> Saved (mIoU {miou:.4f})")

Epoch 1: 100%|██████████| 1771/1771 [04:04<00:00,  7.23it/s]


Epoch 1: Loss 0.8907, Val mIoU 0.3231
  Per-class: ['0.174', '0.298', '0.551', '0.271', '0.322']
  -> Saved (mIoU 0.3231)


Epoch 2: 100%|██████████| 1771/1771 [04:05<00:00,  7.22it/s]


Epoch 2: Loss 0.7264, Val mIoU 0.3325
  Per-class: ['0.184', '0.291', '0.542', '0.302', '0.343']
  -> Saved (mIoU 0.3325)


Epoch 3: 100%|██████████| 1771/1771 [04:05<00:00,  7.22it/s]


...

Epoch 8: Loss 0.6595, Val mIoU 0.3549
  Per-class: ['0.211', '0.322', '0.560', '0.315', '0.367']


Epoch 9: 100%|██████████| 1771/1771 [04:04<00:00,  7.24it/s]


Epoch 9: Loss 0.6536, Val mIoU 0.3572
  Per-class: ['0.197', '0.311', '0.584', '0.332', '0.363']


Epoch 10:   7%|▋         | 117/1771 [00:17<04:00,  6.88it/s]


KeyboardInterrupt: 

In [17]:
# Create test dataset (dummy part labels, not used)
dummy_parts = [np.zeros(len(s['full_shape']['coordinate']), dtype=np.int64) for s in test_split]
test_dataset = MultiTaskAffordanceDataset(test_split, SELECTED_INDICES, dummy_parts,
                                          num_points=2048, occlusion=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

model.load_state_dict(torch.load("best_multitask.pth", map_location=device))
model.eval()
iou_c = np.zeros(5); cnt_c = np.zeros(5)

for pts, aff, _ in tqdm(test_loader, desc="Zero‑Shot Test"):
    pts = pts.permute(0,2,1).to(device); aff = aff.to(device)
    with torch.no_grad():
        aff_logits, _ = model(pts)
        pred = (torch.sigmoid(aff_logits) > 0.5).cpu().numpy()
    aff = aff.cpu().numpy()
    for b in range(pred.shape[0]):
        for c in range(5):
            if aff[b,:,c].sum()>0:
                inter = (pred[b,:,c] * aff[b,:,c]).sum()
                union = (pred[b,:,c] + aff[b,:,c]).clip(0,1).sum()
                iou = inter / (union + 1e-8)
                iou_c[c] += iou; cnt_c[c] += 1

avg_iou = iou_c / (cnt_c + 1e-8)
mean_iou = avg_iou.mean()
print(f"\nZero‑Shot mIoU: {mean_iou:.4f}")
for i, name in enumerate(SELECTED_CLASSES):
    print(f"  {name}: {avg_iou[i]:.4f}")

Zero‑Shot Test: 100%|██████████| 43/43 [00:04<00:00, 10.18it/s]


Zero‑Shot mIoU: 0.0003
  move: 0.0000
  support: 0.0000
  sittable: 0.0000
  contain: 0.0000
  displaY: 0.0017
